In [ ]:
import numpy as np
import math
from scipy.sparse.linalg import expm_multiply
from mpmath import mp
from scipy.sparse import save_npz,load_npz

from sub_function_2D import *

### Parameter Settings

In [ ]:
# Time step size
tau = 5000
# Number of time steps
Total_steps = 100
# Total time
T = Total_steps*tau
# Number of spatial grid points
num_grid = 20
# Spatial step size
delta_x = 1
delta_y = 1
# Scale normalization coefficient
Lambda = 10**0
# Number of variables
N = 5*num_grid**2
# Upper limit of total particle number
m = 2
# Size of Hamiltonian matrix
M = math.comb(m+N, m)
# Number of x qubits
n_x = math.floor(np.log2(M))+1
# Number of ancilla qubits required for U
n_a = 1
eta=1
T_real=Total_steps*tau/(192*eta*(m/2)**(5/2))

mp.dps = 10  # Set precision to n digits

k = 2*np.pi/(num_grid*delta_x)
x_min = -num_grid*delta_x/2
x_max = num_grid*delta_x/2
y_min = -num_grid*delta_x/2
y_max = num_grid*delta_x/2

In [ ]:
# Parameters of KH instability
q = -1      
mass = 1     
epsilon_0 = 1
mu_0 = 1 
B_0 = 2#10**3#2    
density = 1          
u0 = 1       
k_x = 0.99
k_y = 0.2

epsilon = u0/10

### Initial Variable Preparation
$
\bold{x} = \begin{bmatrix}
            u(0,0) \\
            u(\Delta x,0) \\
            u(2\Delta x,0) \\
            \vdots \\
            E(0,0) \\
            E(\Delta x,0) \\
            E(2\Delta x,0) \\
            \vdots 
\end{bmatrix}
$

In [ ]:
# u list
ux = np.zeros((num_grid,num_grid,Total_steps+1))
uy = np.zeros((num_grid,num_grid,Total_steps+1))
# Set initial conditions
# Add perturbation to velocity field
ux[:,:,0] = -u0 * np.ones((num_grid,num_grid))
for i in range(num_grid):
    for j in range(num_grid):
        if int(num_grid/2-3*num_grid/16) <= j <= int(num_grid/2+3*num_grid/16):
            ux[i, j] = u0 + epsilon * np.sin(k_x * i*delta_x+k_y*j*delta_y)
            
# E, B lists
Ex = np.zeros((num_grid,num_grid,Total_steps+1))
Ey = np.zeros((num_grid,num_grid,Total_steps+1))
Bz = np.zeros((num_grid,num_grid,Total_steps+1))
Bz[:,:,0] = B_0 * np.ones((num_grid,num_grid))
# Convert to 1D array
ux_flatten = ux[:,:,0].ravel()
uy_flatten = uy[:,:,0].ravel()
Ex_flatten = Ex[:,:,0].ravel()
Ey_flatten = Ey[:,:,0].ravel()
Bz_flatten = Bz[:,:,0].ravel()
# Initial state preparation
x = np.concatenate([ux_flatten, uy_flatten, Ex_flatten, Ey_flatten, Bz_flatten])
normalize_matrix(x)

### Initial State Preparation
$
|\psi(\bold{x},0)\rangle = \begin{bmatrix}
                    \psi_{m=0} \\
                    cu(0,0) \\
                    cu(\Delta x,0) \\
                    cu(2\Delta x,0) \\
                    \vdots \\
                    cE(0,0) \\
                    cE(\Delta x,0) \\
                    cE(2\Delta x,0) \\
                    \vdots \\
                    \psi_{m=2}
\end{bmatrix},
\|\psi(\bold{x},0)\| = constant
$

### KvN-expm Hamiltonian Simulation Time Evolution
Sequential time evolution with small time step $\tau$  
$|\psi(\bold{x},t+1) \rangle = \exp(-i\frac{H}{\alpha}\tau)|\psi(\bold{x},t) \rangle$

In [ ]:
H = Hamiltonian_matrix_2D_sparse(delta_x, delta_y, Lambda, density, epsilon_0, mu_0, mass, q, m, num_grid)
H, alpha = normalize_matrix_sparse(H)
from scipy.sparse import save_npz
# Save H
save_npz("output/normalized_H_matrix_ng{}.npz".format(num_grid), H)
np.savez("output/alpha_value_U_ng{}.npz".format(num_grid), alpha=alpha)

In [ ]:
# Load H
H = load_npz("output/normalized_H_matrix_ng{}.npz".format(num_grid))
data = np.load("output/alpha_value_U_ng{}.npz".format(num_grid))  # Load the `.npz` file
alpha_loaded = data["alpha"]

In [ ]:
psi = np.zeros((M,Total_steps+1),dtype=np.complex128)
psi[:,0], norm_a, norm_n = state_preparation_sparse_mp_expm(N, M, x, Lambda, m)

In [ ]:
pi_Nby4 = mp.power(np.pi,(N/4))
pi_n = float(mp.floor(mp.log10(pi_Nby4)))
pi_a = float(pi_Nby4 / mp.power(10,pi_n))

psi_now = psi[:,0]

for t in range(1,Total_steps+1):
    psi[:,t] = expm_multiply(-1j * H * 2*tau, psi_now)
    data = psi[:,t] * norm_a / Lambda * pi_a * 10**(norm_n+pi_n) / 2**(1/2)
    for i in range(num_grid):
        for j in range(num_grid):        
            ux[i, j, t] = data[j + i * num_grid+1].real
            uy[i, j, t] = data[j + i * num_grid + num_grid**2+1].real
            Ex[i, j, t] = data[j + i * num_grid + 2*num_grid**2+1].real
            Ey[i, j, t] = data[j + i * num_grid + 3*num_grid**2+1].real
            Bz[i, j, t] = data[j + i * num_grid + 4*num_grid**2+1].real

    # Convert to 1D array
    ux_flatten = ux[:,:,t].ravel()
    uy_flatten = uy[:,:,t].ravel()
    Ex_flatten = Ex[:,:,t].ravel()
    Ey_flatten = Ey[:,:,t].ravel()
    Bz_flatten = Bz[:,:,t].ravel()
    # Initial state preparation
    y = np.concatenate([ux_flatten, uy_flatten, Ex_flatten, Ey_flatten, Bz_flatten])
    psi_now, norm = normalize_matrix(psi[:,t])
    psi_now = psi[:,t]
    print("steps={}".format(t))

In [ ]:
# Save in binary format
filename = 'output/CaseD/2DKelvin-Helmholtz_ux_expm_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, ux)
filename = 'output/CaseD/2DKelvin-Helmholtz_uy_expm_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, uy)
filename = 'output/CaseD/2DKelvin-Helmholtz_Ex_expm_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, Ex)
filename = 'output/CaseD/2DKelvin-Helmholtz_Ey_expm_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, Ey)
filename = 'output/CaseD/2DKelvin-Helmholtz_Bz_expm_numgrid_{}_nx_{}_delta_x_{}_T_{}_delta_t_{}_m_{}.npy'.format(num_grid,n_x, delta_x, T, tau, m)
np.save(filename, Bz)